# Ejercicio 5: Espacio Vectorial

### Leandro Bravo

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

In [ ]:
#Librerías
import os
import pandas as pd
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent
prepro_dir = project_root / '05prepro'
if str(prepro_dir) not in sys.path:
    sys.path.insert(0, str(prepro_dir))

from prepro_func import ensure_nltk_resources, remove_special_characters, tokenize, stemming_tokens
ensure_nltk_resources()

from prepro_func import build_tfidf_matrix, score_queries_tfidf

from bm25_model import build_bm25_index, bm25_score_doc, bm25_rank, score_queries_bm25



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: C:\Users\leand\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [2]:
#Carga del corpus

path = os.path.normpath(r'C:\Users\leand\Desktop\ir2026\ir-2026-BLeandro\ir26a-main\data')
corpus_dir = os.path.join(path, 'wikipedia_text_corpus.csv')

with open(corpus_dir, 'r', encoding='utf-8') as f:
    corpus = f.read()
print(corpus[:100])  


,text
1,"Anovo

Anovo (formerly A Novo) is a computer services company based in Beauvais, France. It


In [3]:

df_corpus = pd.DataFrame({'corpus': [corpus]})
df_corpus

,corpus
0,",text\n1,""Anovo\n\nAnovo (formerly A Novo) is ..."


In [4]:
# Preprocesamiento usando funciones de 05prepro
df_corpus['clean_text'] = df_corpus['corpus'].apply(remove_special_characters)
df_corpus['tokens'] = df_corpus['clean_text'].apply(lambda x: tokenize(x, language='english', remove_stopwords=True))
df_corpus['stemmed'] = df_corpus['tokens'].apply(lambda x: ' '.join(stemming_tokens(x, language='english')))

df_corpus[['clean_text', 'tokens', 'stemmed']]


,clean_text,tokens,stemmed
0,text\n1Anovo\n\nAnovo formerly A Novo is a com...,"[text, anovo, formerly, novo, computer, servic...",text anovo former novo comput servic compani b...


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [5]:
# Construir representación TF-IDF de los documentos preprocesados
vectorizer, tfidf_matrix = build_tfidf_matrix(df_corpus['stemmed'].tolist())

# Definir las 10 queries
queries = [
    'history of science and technology',
    'famous places in europe',
    'machine learning algorithms',
    'artificial intelligence and language models',
    'local flora and fauna',
    'economic development trends',
    'space exploration missions',
    'natural disasters and recovery',
    'environmental sustainability practices',
    'cultural heritage and traditions',
]

# Obtener los puntajes de recuperación
scores = score_queries_tfidf(vectorizer, tfidf_matrix, queries)

result_rows = []
for query_idx, query in enumerate(queries):
    for doc_idx, score in enumerate(scores):
        result_rows.append({
            'query': query,
            'doc_index': doc_idx,
            'score': score[query_idx],
        })

results_df = pd.DataFrame(result_rows)
results_df = results_df.sort_values(by=['query', 'score'], ascending=[True, False])
results_df

,query,doc_index,score
3,artificial intelligence and language models,0,0.000000
9,cultural heritage and traditions,0,0.000000
5,economic development trends,0,0.000491
8,environmental sustainability practices,0,0.000000
1,famous places in europe,0,0.002597
0,history of science and technology,0,0.000016
4,local flora and fauna,0,0.012361
2,machine learning algorithms,0,0.000000
7,natural disasters and recovery,0,0.000000
6,space exploration missions,0,0.035230


## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [13]:
# Construir representación BM25 de los documentos preprocesados

bm25_index = build_bm25_index(df_corpus['stemmed'].tolist())

# Obtener los puntajes de recuperación usando BM25
bm25_results_df = score_queries_bm25(
    queries=queries,
    documents=df_corpus['stemmed'].tolist(),
    index=bm25_index
)
bm25_results_df = pd.DataFrame(bm25_results_df)
bm25_results_df

,query,doc_index,score
0,artificial intelligence and language models,0,0.000000
1,cultural heritage and traditions,0,0.000000
2,economic development trends,0,0.703223
3,environmental sustainability practices,0,0.000000
4,famous places in europe,0,1.383522
5,history of science and technology,0,0.698656
6,local flora and fauna,0,2.070152
7,machine learning algorithms,0,0.000000
8,natural disasters and recovery,0,0.000000
9,space exploration missions,0,0.718978


## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [15]:
#Comparar resultados TF-IDF y BM25
comparison_df = pd.merge(
    results_df,
    bm25_results_df,
    on=['query', 'doc_index'],
    how='outer',
    suffixes=('_tfidf', '_bm25')
)
comparison_df = comparison_df.sort_values(by=['query', 'score_bm25'], ascending=[True, False])
comparison_df

,query,doc_index,score_tfidf,score_bm25
0,artificial intelligence and language models,0,0.000000,0.000000
1,cultural heritage and traditions,0,0.000000,0.000000
2,economic development trends,0,0.000491,0.703223
3,environmental sustainability practices,0,0.000000,0.000000
4,famous places in europe,0,0.002597,1.383522
5,history of science and technology,0,0.000016,0.698656
6,local flora and fauna,0,0.012361,2.070152
7,machine learning algorithms,0,0.000000,0.000000
8,natural disasters and recovery,0,0.000000,0.000000
9,space exploration missions,0,0.035230,0.718978
